# fibonacci_arbitrary_erasure_code_design

Fibonacci inflation **`1 → 10, 0 → 1`** を固定し、logical seed pairを構造的特徴量で選び、任意消失への訂正性能を検証する研究Notebookです。

## 1. 研究目的

目標は、既存baselineより任意消失の `R_KL_arbitrary` が高く、連続消失の `R_KL_contiguous` を維持する **candidate** を見つけ、その構造を説明することです。探索で改善が見つからない場合も保存します。

**本計算前に設定するもの：** 既存研究のbaseline、seed組成、消失条件、過去結果の保存先。baselineや解析①〜⑥の実データは今回の添付に含まれないため、代わりの研究値は設定していません。`DEMO_MODE=True` は検証専用の別設定です。

## 2. 解析①〜⑥との接続

recoverability/KL、finite-size、information leakage、collision、sequence featuresという解析軸を統合します。過去の結論は提供されていないため、相関の符号や設計原理は**仮説**として検証します。過去CSVは冒頭の `PREVIOUS_RESULT_DIRS` で指定します。本実装と計算条件・モデル・サンプルが一致する記録は再利用し、定義が不明な旧形式は参照用として一覧に残します。

## 3. 設計方針と定義

`候補生成 → collision/sequence screening → Pareto/複数ranking → Top-Kのtraining KL → 選択固定 → 独立validation → leakage/有限サイズの比較`

仕様と一致する有限巡回構成の出典は [Li–Boyle, Appendix E.1, Eq. (24)](https://arxiv.org/html/2311.13040v2#A5) です。採用集合のどのseedについても、巡回境界を含む1回の隣接 `01 ↔ 10` 交換後の巡回クラスが採用集合に戻らない条件を実装します。自分自身の巡回クラスへの遷移も検査します。意図する元論文が異なる場合は、この定義を先に照合してください。

同じ長さ・0/1の個数を持つ巡回クラスを使い、反転同値ではまとめません。logical labelの交換は重複除去します。logical stateは**相異なる巡回列の等重み・正振幅の重ね合わせ**です。周期seedでも正規化が正しくなるよう、orbitの重複を除きます。

## 4. 手動入力セル

================================  
ここはユーザーが手動入力する  
================================

初回は同じフォルダで `python -m pip install -r requirements.txt` を実行し、Jupyterで開いてください。`fibonacci_design/` をNotebookと同じ場所に置きます。

`DEMO_MODE=False` のままbaselineを入力すると本設定が使われます。`True` では長さ8の例示pairと小さいサンプルに全体を切り替え、出力先も `demo_run/` に変えます。デモのbaselineは既存研究のbaselineを意味しません。

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
from fibonacci_design.design import Config, Experiment, FEATURE_DIRECTIONS
from fibonacci_design.core import validate_pair, adjacent_swap_valid
from fibonacci_design.report import make_figures
import matplotlib.pyplot as plt

# ==========================================
# ユーザーが手動入力する解析パラメータ
# ==========================================
DEMO_MODE = False

BASELINE_SEED_0 = "..."
BASELINE_SEED_1 = "..."
SEED_LENGTH = 6
K0, K1 = 3, 3

CANDIDATE_MODE = "all"       # all / manual / loaded / random
MANUAL_CANDIDATE_PAIRS = []
LOADED_CANDIDATES_CSV = ""   # seed_0, seed_1 列が必要
PREVIOUS_RESULT_DIRS = []    # 例: ["../fibonacci_collision_analysis/results"]

SCREENING_INFLATION_N = 5
VALIDATION_INFLATION_LIST = [6, 7, 8]
INFLATION_LIST = sorted(set([SCREENING_INFLATION_N] + VALIDATION_INFLATION_LIST))
ERASURE_DOMAIN_MODE = "full_ring"  # full_ring / window
WINDOW_SIZE_LIST = [20, 30, 50]
ERASURE_SIZE_LIST = [1, 2, 3, 4, 5, 6]

MAX_EXACT_PATTERNS = 100000
TRAIN_NUM_SAMPLES = 5000
VALIDATION_NUM_SAMPLES = 10000
TRAIN_RANDOM_SEED = 42
VALIDATION_RANDOM_SEED = 31415
TOP_K_CANDIDATES = 20  # baselineは別枠。ablation/Paretoを含め合計K候補
TOP_K_LEAKAGE = 5     # baselineは別枠。training結果だけで選び固定

LOW_WEIGHT_THRESHOLD = 5
MAX_CONTIGUOUS_DEGRADATION = 0.02
REQUIRE_NO_CONTIGUOUS_DEGRADATION = False
MAX_LEAKAGE_DEGRADATION = 1e-9
REQUIRE_STATISTICAL_EVIDENCE = True
MIN_VALIDATION_SIZES = 2
MIN_IMPROVING_M = 2

# Leakageだけをさらに間引く場合、観測最大値と真の最大値を区別します。
LEAKAGE_NUM_SAMPLES = 200
MAX_EXACT_LEAKAGE_PATTERNS = 2000
MAX_LEAKAGE_DIMENSION = 2048

NORMALIZATION = "minmax"  # minmax / zscore
WEIGHT_D_CROSS = 1.0
WEIGHT_LOW_SUPPORT = 1.0
WEIGHT_EXPECTED_COLLISION = 1.0
WEIGHT_LONG_RANGE = 1.0
WEIGHT_CYCLIC_SPAN = 0.5
WEIGHT_DANGEROUS_MULTIPLICITY = 0.5
WEIGHT_LOCAL_SUBSTRING = 0.5
WEIGHT_SHORT_RANGE = 0.5
WEIGHT_C3 = 1.0

SUBSTRING_LENGTHS = (2, 3, 4)
SHORT_LAGS = (1, 2, 3)
LONG_LAGS = None   # 自動: N//4, N//3, N//2 の有効なlag
C3_OFFSETS = None # 自動: (1,2), (1,N//2), (N//3,2*N//3)

MAX_RAW_SEEDS = 1000000
MAX_POSSIBLE_PAIRS = 2000000
RANDOM_PAIR_COUNT = 500
RANDOM_MAX_ATTEMPTS = 100000
N_JOBS = 1        # まずserialで検証。初回の並列screeningはRESUME=False
RESUME = True
SAVE_RESULTS = True
SAVE_FIGURES = True
OUTPUT_DIR = Path(".")

================================  
ここから下は基本的に自動計算  
================================

baseline未入力なら次セルで説明付きで停止します。過去データのない項目を推測して埋めることはありません。デモは動作検証と使用例です。

In [ ]:
config = Config(
    baseline_seed_0=BASELINE_SEED_0, baseline_seed_1=BASELINE_SEED_1,
    seed_length=SEED_LENGTH, k0=K0, k1=K1, candidate_mode=CANDIDATE_MODE,
    manual_candidate_pairs=MANUAL_CANDIDATE_PAIRS, loaded_candidates_csv=LOADED_CANDIDATES_CSV,
    previous_result_dirs=PREVIOUS_RESULT_DIRS,
    screening_inflation_n=SCREENING_INFLATION_N, validation_inflation_list=VALIDATION_INFLATION_LIST,
    erasure_domain_mode=ERASURE_DOMAIN_MODE, window_size_list=WINDOW_SIZE_LIST,
    erasure_size_list=ERASURE_SIZE_LIST, max_exact_patterns=MAX_EXACT_PATTERNS,
    train_num_samples=TRAIN_NUM_SAMPLES, validation_num_samples=VALIDATION_NUM_SAMPLES,
    train_random_seed=TRAIN_RANDOM_SEED, validation_random_seed=VALIDATION_RANDOM_SEED,
    top_k_candidates=TOP_K_CANDIDATES, top_k_leakage=TOP_K_LEAKAGE,
    low_weight_threshold=LOW_WEIGHT_THRESHOLD,
    max_contiguous_degradation=MAX_CONTIGUOUS_DEGRADATION,
    require_no_contiguous_degradation=REQUIRE_NO_CONTIGUOUS_DEGRADATION,
    max_leakage_degradation=MAX_LEAKAGE_DEGRADATION,
    require_statistical_evidence=REQUIRE_STATISTICAL_EVIDENCE,
    min_validation_sizes=MIN_VALIDATION_SIZES, min_improving_m=MIN_IMPROVING_M,
    leakage_num_samples=LEAKAGE_NUM_SAMPLES, max_exact_leakage_patterns=MAX_EXACT_LEAKAGE_PATTERNS,
    max_leakage_dimension=MAX_LEAKAGE_DIMENSION, normalization=NORMALIZATION,
    weights={"d_cross":WEIGHT_D_CROSS,"ell_cross_min":WEIGHT_CYCLIC_SPAN,
             "low_weight_cross_support_fraction":WEIGHT_LOW_SUPPORT,
             "dangerous_support_multiplicity":WEIGHT_DANGEROUS_MULTIPLICITY,
             "expected_cross_collision_arbitrary":WEIGHT_EXPECTED_COLLISION,
             "local_substring_mismatch":WEIGHT_LOCAL_SUBSTRING,
             "short_range_C2_mismatch":WEIGHT_SHORT_RANGE,
             "long_range_C2_mismatch":WEIGHT_LONG_RANGE,"C3_mismatch":WEIGHT_C3},
    substring_lengths=SUBSTRING_LENGTHS, short_lags=SHORT_LAGS,
    long_lags=LONG_LAGS, c3_offsets=C3_OFFSETS,
    max_raw_seeds=MAX_RAW_SEEDS, max_possible_pairs=MAX_POSSIBLE_PAIRS,
    random_pair_count=RANDOM_PAIR_COUNT, random_max_attempts=RANDOM_MAX_ATTEMPTS,
    output_dir=str(OUTPUT_DIR), save_results=SAVE_RESULTS, save_figures=SAVE_FIGURES,
    resume=RESUME, n_jobs=N_JOBS,
)
if DEMO_MODE:
    config = Config(
        baseline_seed_0="00001111", baseline_seed_1="01010101", seed_length=8, k0=4, k1=4,
        screening_inflation_n=0, validation_inflation_list=[1,2,3], erasure_size_list=[1,2,3,4],
        max_exact_patterns=150, train_num_samples=100, validation_num_samples=150,
        top_k_candidates=6, top_k_leakage=3, leakage_num_samples=40,
        max_exact_leakage_patterns=80, output_dir="demo_run", resume=True,
    )
    display(Markdown("**DEMO：例示baseline・小規模サンプルを使用しています。研究結果の代用ではありません。**"))
config.validate()
experiment = Experiment(config)

## 5. baseline検証

長さ・組成・巡回非同値・隣接交換条件を検査します。baselineも候補と同じN、W、m、消失パターンで評価します。異なる長さのbaselineとの比較は行いません。

In [ ]:
display(pd.DataFrame([experiment.baseline]))
display(experiment.tables["prior_inventory"])

## 6. valid seed生成

指定組成を持つ全列から巡回canonical representativeを作ります。`all` で空間が大きすぎる場合は明示的に停止します。`random` は固定組成の列を抽出して制約を再検査する探索で、巡回クラス一様抽出や全探索ではありません。mutation/heuristicによる規則変更は実装していません。

## 7. valid logical pair生成

相異なるrepresentativeのunordered pairを検査します。manual/loadedでも同じ検査を通します。除外された入力は理由付きで保存します。

## 8. candidate数確認

**長さ6・K0=K1=3では、20列 → 4巡回クラス → 6組 → 1 valid pair (`000111`, `010101`) です。** baselineがこの組なら、同じ探索空間に別候補はありません。この場合、候補を捏造せずnegative resultを保存します。長さ8・4/4では27組、長さ10・5/5では261組となることを別の全列挙テストで検証しています。

In [ ]:
display(pd.DataFrame([experiment.counts]))
display(experiment.candidates)
if not experiment.rejected.empty:
    display(experiment.rejected)
if experiment.counts["nonbaseline_candidates"] == 0:
    display(Markdown("**代替candidateなし。baseline評価を続け、探索範囲内のnegative resultとして保存します。**"))

## 9. inflation / orbitと計算条件

Nは常に `len(inflate(seed,n))` から生成します。windowは環上の位置 `0,...,W-1`。任意消失はそのm-subset、連続消失はwindow内で折り返さない区間です。full ringでは境界を跨ぐ連続区間も含めます。W=Nはfull ringと同じ定義です。

W>N、m>Wは計算不能な条件として除外し、下のauditに表示します。異なる条件を暗黙に平均せず、CSVにN,W,mを記録します。

In [ ]:
display(experiment.tables["condition_audit"])
# 小さなpilotで、本screeningとKLの時間を見積もる。
display(experiment.benchmark())

## 10. Stage 1：collision feature

$O_a$ は相異なる巡回列の集合、$D=x\oplus y$ はdifference support、$\mu(D)$ はそのmultiplicityです。

- `d_cross`：最小Hamming距離。`ell_cross_min`：supportを覆う最小巡回区間長。
- `low_weight_cross_support_fraction`：重みがthreshold以下の**異なるsupport**の割合。
- `support_multiplicity`：全異なるsupportで平均したmultiplicity。`dangerous_support_multiplicity` はthreshold以下に限定した最大値です。
- `expected_cross_collision_arbitrary`：各mの期待cross orbit-pair collision数を平均。window外へはみ出すsupportの寄与は0です。

$$\mathbb E[C_{cross}(E)]=\sum_{D\subseteq W}\mu(D)\frac{\binom{W-|D|}{m-|D|}}{\binom{W}{m}}.$$

連続消失では、supportを含む区間の正確な割合を用います。単一の最小spanだけから包含確率を近似しません。raw countとorbit-pair数で割ったnormalized countを保存します。複数Wのscreeningでは、各Wでm平均を計算してからW平均を使い、条件別の値も保存します。

`d_cross > m` はcross collisionを防ぎますが、それだけではKLの対角条件を保証しません。期待collision数はKL失敗確率そのものではありません。

In [ ]:
screening = experiment.screen()
collision_columns = ["candidate_id","seed_0","seed_1","N","d_cross","ell_cross_min",
                     "low_weight_cross_support_fraction","number_unique_cross_supports",
                     "support_multiplicity","dangerous_support_multiplicity",
                     "expected_cross_collision_arbitrary","expected_cross_collision_contiguous"]
display(screening[collision_columns].head(20))
display(experiment.tables["collision_spectrum"].head(20))

## 11. Stage 1：sequence feature

substring mismatchは巡回substring分布間のtotal variationを指定長で平均します。spinを $s_i=2b_i-1$ とし、$C_2(r)=N^{-1}\sum_i s_i s_{i+r}$、$C_3(r,t)=N^{-1}\sum_i s_i s_{i+r}s_{i+t}$ のlogical間差の絶対値を指定lagで平均します。local statisticsとnonlocal statisticsは別々に残します。

long lagの既定値はNの割合で決まるため、過去解析が固定lagを使っていた場合は `LONG_LAGS` / `C3_OFFSETS` を揃えてください。小さいNで有効なlagがなくなるfeatureは0で、予測情報を持ちません。

In [ ]:
display(screening[["candidate_id","local_substring_mismatch","short_range_C2_mismatch",
                   "long_range_C2_mismatch","C3_mismatch"]].head(20))

## 12. feature normalization

raw値を保持したままmin-maxまたはz-scoreを計算し、大きいほど期待性能がよい方向へ揃えます。全候補で同じ値のfeatureは0にします。符号は仮説であり、結果から証明された関係ではありません。

## 13. Pareto analysis

`d_cross ↑ / expected collision ↓ / ell_cross_min ↑ / local substring mismatch ↓` の4目的で非劣候補を求めます。単一scoreと異なる選択を残します。

## 14. DesignScore

指定した重みと方向を用いて正規化featureを加算します。重みは探索設定であり、研究成果ではありません。

In [ ]:
display(pd.DataFrame([{"feature":f,"direction":"maximize" if d>0 else "minimize",
                      "weight":config.weights.get(f,0)} for f,d in FEATURE_DIRECTIONS.items()]))
display(screening.loc[screening.pareto_optimal, ["candidate_id","seed_0","seed_1","design_score","d_cross","ell_cross_min"]])

## 15. Top-K screening

Pareto優先、combined、collision only、sequence only、leave-one-feature-outの順位から順番に候補を取り、重複を除いて合計K候補を選びます。baselineは別枠です。Kが小さいと全方式のTop候補を詳細評価できないことがあり、未評価のablation性能は欠損値のまま残します。

In [ ]:
display(screening[["candidate_id","seed_0","seed_1","design_score","collision_score","sequence_score",
                   "pareto_optimal","selected_for_KL","is_baseline"]].head(20))
print("KL evaluation IDs:", experiment.selected_ids)

## 16. KL benchmarkと正確判定

pilot時間は前掲のbenchmarkを参照してください。Nやcollision構造、固有値計算で時間は変わります。既定の本設定は重くなる可能性があるため、保存・再開を利用できます。

消失領域E、保持領域Bについて
$$C_{ab}(e,f)=\#\{(x,y)\in O_a\times O_b:x_B=y_B,\ x_E=e,\ y_E=f\},\qquad
\rho_E^{ab}=\frac{C_{ab}}{\sqrt{|O_a||O_b|}}.$$
正確なKL条件は
$$C_{01}=0,\qquad |O_1|C_{00}=|O_0|C_{11}.$$
**整数比較**なので、KL判定に数値toleranceは使いません。同一orbit内のcollisionによる非対角成分も計算します。$2^N$次元の状態ベクトルは本計算で作りません。

`R_rec` は **O0∪O1の列を保持bitから一意に復元できるか**という古典的指標です。`R_label_rec` はlogical labelだけの識別可能性です。完全な量子recoverabilityはKLと同値であり、この古典的指標と混同しません。[Knill–Laflamme](https://arxiv.org/abs/quant-ph/9604034)

巡回orbit stateは平行移動不変なので、各固定N,mの連続消失訂正率は0または1です。位置0の計算は全開始位置の厳密評価に等価で、`exact_translation_symmetry` と記録します。

## 17. Stage 2：training KL validation

任意消失は可能なら全列挙します。超過時は重複なし一様抽出で、全candidateに同じmaskを使います。mask一覧とhashを保存します。KLの各sample結果を保存し、baselineとの差を同じpatternで比較します。

## 18. baseline比較

`Delta_R_arbitrary` と `Delta_R_contiguous` を条件別に計算します。sampledのrateにWilson区間、paired differenceに保守的なHoeffding区間を併記します。これは各条件の探索的区間で、最終判定では比較数を考慮した別の保守的な基準を使います。

In [ ]:
training = experiment.train()
performance_columns=["candidate_id","inflation_n","N","window_size","erasure_size","R_KL_arbitrary",
                     "R_KL_contiguous","R_rec_arbitrary","Delta_R_arbitrary","Delta_R_contiguous",
                     "calculation_mode","n_patterns","paired_delta_ci_low","paired_delta_ci_high"]
display(training[performance_columns])
display(pd.DataFrame([experiment.frozen_selection]))

## 19. Stage 3：information leakage

trainingの連続性能制約を満たす候補を平均改善量で並べ、少数候補を**validationを見る前に固定**します。
$$\rho_{RE}=\tfrac12\begin{pmatrix}\rho^{00}&\rho^{01}\\\rho^{10}&\rho^{11}\end{pmatrix},\quad
I(R:E)=1+S((\rho^{00}+\rho^{11})/2)-S(\rho_{RE})$$
はbits単位です。$D_Z=\frac12\|\rho^{00}-\rho^{11}\|_1$、$D_X=\frac12\|\rho^{01}+\rho^{10}\|_1$ を使用し、補助的にY基底も保存します。$D_X$はX基底のtrace distanceという定義です。

全candidateでI(R:E)を計算しません。大きい条件ではKL用maskの共通部分集合を一様に抽出します。`max_I_RE` は sampled の場合 **観測最大値（真の最大値の下界）** です。`worst_case_certified=True` は全patternを列挙できた条件だけに付きます。KLが高いという理由だけでworst-case leakageを証明したとは扱いません。

In [ ]:
leakage_train = experiment.leakage(experiment.leakage_ids,[config.screening_inflation_n],"train")
display(leakage_train[["candidate_id","N","window_size","erasure_size","mean_I_RE","median_I_RE",
                       "max_I_RE","mean_D_Z","max_D_Z","mean_D_X","max_D_X","leakage_mode","worst_case_certified"]])

## 20. multiple-m validation

候補選択を固定した後、各mを検証します。m平均、mに対する台形積分を区間長で割ったAUC、最小訂正率を併記します。AUCはN,Wごとに計算してから平均し、異なるサイズのm点を1本の曲線として積分しません。

## 21. multiple-N validation

screeningと同じNの別splitに加え、指定した全validation nでKL・collision・leakageを再評価します。最終合否には `VALIDATION_INFLATION_LIST` の条件だけを使います。少なくとも2サイズ、複数mでの改善が必要です。

window modeでは固定WのままNを増やせます。ただし、有限のN点から `R→1` や `I→0` という無限系の結論は出しません。

## 22. train / validation sample比較

sampledではtraining maskをvalidationから除外し、重なりのない一様標本を作ります。validationの母集団は「trainingを除く残余集合」で、`effective_population` に記録します。全列挙可能なら両splitは同じ全集合です。この場合は独立試験とは呼ばず、正確な計算結果として優先します。

In [ ]:
validation = experiment.validate()
display(experiment.tables["aggregate"])
display(validation[performance_columns])
split_check = experiment.tables["performance"].groupby(
    ["N","window_size","erasure_size","sample_split"],as_index=False
).agg(sample_sets=("sample_id","nunique"),n_patterns=("n_patterns","first"),
      effective_population=("effective_population","first"),mode=("calculation_mode","first"))
assert (split_check.sample_sets == 1).all(), "Candidates used different samples"
display(split_check)

## 23. feature ablationと予測精度

collision only、sequence only、combined、featureを1つずつ外した順位を保存します。実際にKL評価した候補にはtraining性能を結合し、未評価候補は欠損値にします。相関は**選別された候補内の探索的相関**であり、全候補に対する予測精度ではありません。候補数が少ない、featureまたはKL値が一定の場合は相関を算出しません。

無偏りの精度検証が必要な次段階では、独立candidate holdoutまたは全候補のKL評価が必要です。現NotebookではTop-Kの計算制限を守ります。

In [ ]:
ablation = experiment.tables["ablation"]
display(ablation[ablation["rank"]<=3])
display(experiment.tables["correlations"])

## 24. final candidate ranking

合否は、隣接条件・指定サイズのcoverage・複数N/mでの改善・各条件で任意性能の観測上の非悪化・連続性能制約・leakage coverageと観測最大値の非悪化を確認します。sampledでは既定で比較数を考慮した保守的なpaired改善の証拠も要求します。leakage未計算candidateは合格にしません。

A：複数サイズ・mで任意性能改善、連続性能非悪化。B：改善はあるが連続性能が許容範囲で低下。C：検証で改善なし。D：特定条件だけ改善、または必要な広がりを満たさない。A/Bであってもleakageや統計的証拠が不足すれば `validation_pass=False` です。

**validation_passは、この有限の実験設定における候補判定です。sampled最大漏洩のglobal保証や新しいQECCの証明を意味しません。** collision/sequenceの整合は設計仮説の検討材料で、重みの都合で合否に追加しません。

In [ ]:
final_candidates = experiment.tables["final"]
display(final_candidates)
if final_candidates.empty:
    display(Markdown("**この探索範囲にはbaseline以外の詳細評価候補がありません。**"))
else:
    assert all(adjacent_swap_valid(r.seed_0,r.seed_1) for r in final_candidates.itertuples())
    display(final_candidates[["rank","candidate_id","classification","validation_pass",
                              "average_arbitrary_improvement","worst_arbitrary_improvement","contiguous_change",
                              "improving_size_count","improving_m_count","max_I_RE","baseline_comparison"]])

## 25. Baseline / Candidate 1〜3 比較表

性能列は指定validation条件の平均、collision/sequence列はscreening nの値です。`gap_KL = R_KL_contiguous - R_KL_arbitrary`。条件別の厳密な比較はperformance/validation_features表を参照してください。max I_REは観測した全validation条件での最大値です。

In [ ]:
display(experiment.tables["comparison"])

## 26. グラフ

featureと性能の散布図は、screening n・training・共通W,mで比較します。性能曲線はN,WまたはW,mごとにパネルを分けます。full ringではW=Nであることを明記します。最大3候補とbaselineを表示し、図だけで異なる実験条件を混同しないようにします。

In [ ]:
figures = make_figures(experiment.tables, Path(config.output_dir)/"figures",save=config.save_figures)
for name, figure in figures.items():
    display(Markdown(f"**{name}**"))
    display(figure)
    plt.close(figure)

## 27. 最終出力と保存

`results/` に候補、screening、KL、leakage、最終順位、baseline比較、collision spectrum、ablation、相関、benchmark、条件auditをCSV保存します。seed列を再読込するときは必ず `dtype={"seed_0": str, "seed_1": str}` を指定してください。

各candidate/条件ごとのJSON checkpointを原子的に保存し、同じ設定で再実行すると再利用します。sample一覧、manifest、validation前の選択も保存します。モデル・条件・mask hashを含むcache keyにより異なる実験の結果を混ぜません。`SAVE_RESULTS=False` では書き込みません。

旧CSVに必要な由来情報がない場合は自動的に信用せず参照用にします。元Notebookのモデル、正規化、領域定義、sample集合を確認してから変換してください。大規模研究用にserialを既定とし、独立なfeature taskはmultiprocessing可能です。

In [ ]:
experiment.save()
display(pd.DataFrame([experiment.conclusion]))
print("Output directory:", Path(config.output_dir).resolve())
if config.save_results:
    for path in sorted((Path(config.output_dir)/"results").glob("*.csv")):
        print(path.name)

## 28. 結果の解釈：設計原理を検討する

Q1/Q8はKL改善と連続性能、Q2〜Q5はfeature・相関・ablation、Q6/Q7はN,m曲線で評価します。次セルでは候補ごとの差を並べます。

検証する仮説は「低weight support減少 → cross collision減少 → off-diagonal leakage低下 → KL改善」です。しかし対角KL不一致も独立の失敗原因です。相関や同時改善だけから因果関係を断定せず、反例も残します。local/long-range featureが変わらない場合、そのfeatureでは改善を説明できません。

sampledで有望でも、条件ごとの精度・leakageの標本数・全列挙可能性を確認してください。exactな性能結果は、その有限条件内に限って確定値です。改善なしの場合はseed lengthの拡張を次の実験とし、logical basis・振幅・位相・substitutionの変更は本Notebookの範囲外です。

In [ ]:
base_features = screening[screening.is_baseline].iloc[0]
diagnostics=[]
for row in final_candidates.head(3).itertuples():
    feature = screening[screening.candidate_id==row.candidate_id].iloc[0]
    diagnostics.append({
        "candidate_id":row.candidate_id,
        "Delta_d_cross":feature.d_cross-base_features.d_cross,
        "Delta_low_weight_fraction":feature.low_weight_cross_support_fraction-base_features.low_weight_cross_support_fraction,
        "Delta_expected_collision":feature.expected_cross_collision_arbitrary-base_features.expected_cross_collision_arbitrary,
        "Delta_long_C2":feature.long_range_C2_mismatch-base_features.long_range_C2_mismatch,
        "Delta_C3":feature.C3_mismatch-base_features.C3_mismatch,
        "Delta_max_leakage_worst_condition":row.worst_leakage_change,
        "mean_Delta_R_arbitrary":row.average_arbitrary_improvement,
        "worst_Delta_R_contiguous":row.contiguous_change,
        "improving_N_count":row.improving_size_count,"improving_m_count":row.improving_m_count,
        "validation_pass":row.validation_pass,
    })
display(pd.DataFrame(diagnostics))
if not final_candidates.empty and final_candidates.validation_pass.any():
    display(Markdown("設定した有限の検証条件を通過したcandidateがあります。構造指標の変化と条件別表を照合し、説明できる点・説明できない点を分けて記録してください。"))
else:
    display(Markdown("**現設定で採用基準を通過したcandidateはありません。** 探索空間の制約、改善の条件依存、標本精度、leakage coverageのどれが原因かを理由列で確認してください。"))

### 実装と検証資料

`fibonacci_design/core.py`：隣接条件、正確KL、縮約状態、sampling。`features.py`：collision/sequence。`design.py`：探索・保存・最終判定。`report.py`：図表。

`python -m unittest -v test_fibonacci_design.py` で、小さい系のdense partial trace、全maskのKL、I(R:E)、collision期待値の全列挙、候補数、sampling、再開、CSV再利用を再検証できます。

出典：[有限Fibonacci構成](https://arxiv.org/html/2311.13040v2#A5)、[KL条件](https://arxiv.org/abs/quant-ph/9604034)。数式・surrogate・分類の実装上の定義は上記各節に明示しました。